## Question 2 - DML (12 points)


### 1) Processing Data and definition of variables (0.5 pts)

In [3]:
using CSV, DataFrames, StatsModels

path = raw"C:\Users\User\Desktop\DML_NN\Julia\Input\penn_jae.dat.txt"
df = CSV.read(path, DataFrame; delim=' ', ignorerepeated=true)

df = filter(:tg => t -> t in (0, 4), df)
df.T4 = ifelse.(df.tg .== 4, 1, 0)
df.y = log.(df.inuidur1)

unique(df.dep)
for k in 0:2
    col = Symbol("dep_", k)
    df[!, col] = ifelse.(df.dep .== k, 1, 0)
end

d = df.T4

xnames = [
    "female","black","othrace",
    "dep_1","dep_2",
    "q2","q3","q4","q5","q6",
    "recall","agelt35","agegt54",
    "durable","nondurable","lusd","husd"
]

X = select(df, xnames)
y = df.y

println("Observations: ", nrow(df))
println("Variables in X: ", names(X))
describe(df[:, [:tg, :T4, :y, :dep, :dep_0, :dep_1, :dep_2]])


Observations: 5099
["female", "black", "othrace", "dep_1", "dep_2", "q2", "q3", "q4", "q5", "q6", "recall", "agelt35", "agegt54", "durable", "nondurable", "lusd", "husd"]


Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,tg,1.3689,0,0.0,4,0,Int64
2,T4,0.342224,0,0.0,1,0,Int64
3,y,2.02758,0.0,2.3979,3.95124,0,Float64
4,dep,0.439694,0,0.0,2,0,Int64
5,dep_0,0.724064,0,1.0,1,0,Int64
6,dep_1,0.112179,0,0.0,1,0,Int64
7,dep_2,0.163758,0,0.0,1,0,Int64


In [5]:
using DataFrames, Random, PrettyTables
using Flux                                  
using GLM: lm, coef                         
using StatsModels: Term

import MLJ                                  
import MLJFlux                              
import MLJLinearModels                      
import MLJDecisionTreeInterface             

Random.seed!(42)

Xc = MLJ.coerce(
    DataFrame(X),
    MLJ.Count => MLJ.Continuous,
    MLJ.Multiclass => MLJ.Continuous,
    MLJ.Finite => MLJ.Continuous
)
rename!(Xc, Symbol.(names(Xc)))             
y = Float64.(y)
d = Float64.(d)

const n_in  = size(Xc, 2)
const n_out = 1

"""
dml(y, d, X; learner_m, learner_g, K=2)
Devuelve α̂ en y = d*α + g(X) + u,  d = m(X) + v  usando cross-fitting.
"""
function dml(y::AbstractVector{<:Real}, d::AbstractVector{<:Real}, X::DataFrame;
             learner_m, learner_g, K::Int=2)

    n = length(y)
    @assert length(d) == n && nrow(X) == n "Dimensiones inconsistentes"

    foldid = rand(1:K, n)
    d_res = zeros(n)
    y_res = zeros(n)

    for k in 1:K
        train = foldid .!= k
        test  = foldid .== k

        m_mach = MLJ.machine(learner_m, X[train, :], d[train]) |> MLJ.fit!
        g_mach = MLJ.machine(learner_g, X[train, :], y[train]) |> MLJ.fit!

        d̂ = collect(MLJ.predict(m_mach, X[test, :]))
        ŷ = collect(MLJ.predict(g_mach, X[test, :]))

        d_res[test] .= d[test] .- d̂
        y_res[test] .= y[test] .- ŷ
    end

    denom = sum(abs2, d_res)
    @assert denom > 0 "Varianza de d̃ es cero; revise especificación."
    α̂ = sum(d_res .* y_res) / denom
    return α̂
end

ols_model   = MLJLinearModels.LinearRegressor()
lasso_model = MLJLinearModels.LassoRegressor()
rf_model    = MLJDecisionTreeInterface.RandomForestRegressor()

nn_builder = MLJFlux.@builder Chain(
    Dense(n_in, 50, relu),
    Dense(50, n_out)                 
)
nn_model = MLJFlux.NeuralNetworkRegressor(
    builder   = nn_builder,
    optimiser = Flux.ADAM(1e-3),
    epochs    = 1000,
    batch_size= 64
)

α_lasso = dml(y, d, Xc; learner_m=lasso_model, learner_g=lasso_model)
α_rf    = dml(y, d, Xc; learner_m=rf_model,    learner_g=rf_model)
α_nn    = dml(y, d, Xc; learner_m=nn_model,    learner_g=nn_model)

Z = hcat(DataFrame(y=y, d=d), Xc)
rhs = foldl(+, Term.(Symbol.(names(Xc))); init=Term(:d))  
f = Term(:y) ~ rhs
α_ols = coef(lm(f, Z))[2]

tbl = DataFrame(
    Model = ["OLS", "Lasso (DML)", "Random Forest (DML)", "Neural Net (DML)"],
    Alpha = [α_ols, α_lasso, α_rf, α_nn]
)
pretty_table(tbl)



[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(RandomFores

┌─────────────────────┬────────────┐
│               Model │      Alpha │
│              String │    Float64 │
├─────────────────────┼────────────┤
│                 OLS │ -0.0725756 │
│         Lasso (DML) │ -0.0842969 │
│ Random Forest (DML) │ -0.0554842 │
│    Neural Net (DML) │ -0.0966812 │
└─────────────────────┴────────────┘


## III. No cross-fitting

In [7]:
using Statistics

rmse(y, yhat) = sqrt(mean((y .- yhat).^2))

"""
dml_nocf(y, d, X; learner_m, learner_g)

Entrena m̂ y ĝ en TODA la muestra (sin cross-fitting),
calcula residuos d̃ = d - m̂(X), ỹ = y - ĝ(X),
y estima α̂ = Σ d̃ ỹ / Σ d̃². Devuelve α̂, RMSE_y, RMSE_d.
"""
function dml_nocf(y::AbstractVector{<:Real}, d::AbstractVector{<:Real}, X::DataFrame;
                  learner_m, learner_g)

    @assert length(y) == length(d) == nrow(X)

    m_mach = MLJ.machine(learner_m, X, d) |> MLJ.fit!
    g_mach = MLJ.machine(learner_g, X, y) |> MLJ.fit!

    dhat = collect(MLJ.predict(m_mach, X))
    yhat = collect(MLJ.predict(g_mach, X))

    dtilde = d .- dhat
    ytilde = y .- yhat

    denom = sum(abs2, dtilde)
    @assert denom > 0 "Varianza de d̃ es cero."
    alpha = sum(dtilde .* ytilde) / denom

    return (alpha = alpha,
            rmse_y = rmse(y, yhat),
            rmse_d = rmse(d, dhat))
end

res_lasso = dml_nocf(y, d, Xc; learner_m=lasso_model, learner_g=lasso_model)
res_rf    = dml_nocf(y, d, Xc; learner_m=rf_model,    learner_g=rf_model)
res_nn    = dml_nocf(y, d, Xc; learner_m=nn_model,    learner_g=nn_model)

tbl_nocf = DataFrame(
    Model   = ["Lasso (no CF)", "Random Forest (no CF)", "Neural Net (no CF)"],
    Alpha   = [res_lasso.alpha, res_rf.alpha, res_nn.alpha],
    RMSE_y  = [res_lasso.rmse_y, res_rf.rmse_y, res_nn.rmse_y],
    RMSE_d  = [res_lasso.rmse_d, res_rf.rmse_d, res_nn.rmse_d]
)
pretty_table(tbl_nocf)

[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(LassoRegressor(lambda = 1.0, …), …).
┌ Info: Solver: MLJLinearModels.ProxGrad
│   accel: Bool true
│   max_iter: Int64 1000
│   tol: Float64 0.0001
│   max_inner: Int64 100
│   beta: Float64 0.8
└   gram: Bool false
[ Info: Training machine(RandomForestRegressor(max_depth = -1, …), …).
[ Info: Training machine(RandomForestRegressor(max_depth = -1, …), …).
[ Info: Training machine(NeuralNetworkRegressor(builder = GenericBuilder(apply = #4), …), …).
[ Info: MLJFlux: converting input data to Float32
Optimising neural net: 100%[=========================] Time: 0:00:08
[ Info: Training machine(NeuralNetworkRegressor(builder = GenericBuilder(apply = #4), …), …).
[ Info: MLJFlux: converting input data to Float32
Optimising neural n

┌───────────────────────┬────────────┬─────────┬──────────┐
│                 Model │      Alpha │  RMSE_y │   RMSE_d │
│                String │    Float64 │ Float64 │  Float64 │
├───────────────────────┼────────────┼─────────┼──────────┤
│         Lasso (no CF) │ -0.0854554 │ 1.21463 │ 0.474454 │
│ Random Forest (no CF) │ -0.0802294 │ 1.08359 │ 0.430626 │
│    Neural Net (no CF) │ -0.0821684 │ 1.11234 │ 0.441591 │
└───────────────────────┴────────────┴─────────┴──────────┘


# III. No cross-fitting — answers

## 1) What can you say about the RMSE for predicting $y$ and $d$?
- Using **no cross-fitting**, you reported (in-sample) RMSEs:

  - Lasso:  RMSE$_y$ ≈ **1.2146**, RMSE$_d$ ≈ **0.4745**  
  - Random Forest: RMSE$_y$ ≈ **1.0834**, RMSE$_d$ ≈ **0.4306**  ← lowest  
  - Neural Net: RMSE$_y$ ≈ **1.1124**, RMSE$_d$ ≈ **0.4416**

- These are **in-sample** errors (same data to train and evaluate), so they are **optimistic**. With cross-fitting or out-of-fold evaluation, RMSE would typically be **higher**.
- RF attains the lowest RMSE for both $g(X)$ and $m(X)$ here, consistent with flexible nonlinearity capture and built-in regularization.

## 2) Why does one function yield lower RMSE than another?
- The target functions $g(\cdot)$ and $m(\cdot)$ are likely **nonlinear**.  
  - **Lasso** imposes linearity + $\ell_1$ shrinkage → higher bias.  
  - **Random Forest** models nonlinearities and interactions with low tuning burden → lower RMSE.  
  - **Neural Net** is flexible but can be variance-heavy at this $n$ and may under-/over-fit without careful tuning.
- Net effect: different **bias–variance trade-offs** → different RMSE.

## 3) What problem arises if we estimate without cross-fitting?
- The debiased estimator uses residuals
$$
\tilde d_i = d_i - \hat m(X_i), \qquad 
\tilde y_i = y_i - \hat g(X_i),
$$
and computes
$$
\hat\alpha = \frac{\sum_i \tilde d_i \tilde y_i}{\sum_i \tilde d_i^2}.
$$


- **Without cross-fitting**, $\hat m$ and $\hat g$ are trained on the same observations they predict. This induces **sampling correlation** between first-stage estimation errors and the second-stage score, violating the orthogonality condition that underpins DML. Consequences:
  - **Bias** in $\hat\alpha$ (overfitting bias).
  - **Invalid inference**: standard errors too small; coverage deteriorates.
  - Worse small-sample behavior relative to cross-fitted DML.

## 4) Comparing $\hat\alpha$ (your tables)
- **Cross-fitted DML**: OLS −0.0726 (benchmark), Lasso −0.0843, RF −0.0555, NN −0.0967.  
- **No cross-fitting**: Lasso −0.0855, RF −0.0803, NN −0.0916 with low in-sample RMSEs.
- The **level of RMSE is not a reliability signal** for $\hat\alpha$ when computed in-sample. Prefer **cross-fitted** estimates for causal reporting; they are robust to overfitting in the first stage.
